# Import

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from imblearn.over_sampling import SMOTE

# Pull Data

In [2]:
# Pull labels and store to y
y = np.genfromtxt('data_labels.csv', delimiter=',', dtype=int)
# Pull training data features
X = np.genfromtxt('data.csv', delimiter=',')


In [3]:
print(f'Original shapes')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

Original shapes
X shape: (3486, 354)
y shape: (3486,)


In [4]:
def show_distribution(y):
    unique_y, y_counts = np.unique(y, return_counts=True)
    total_counts = y_counts.sum()
    for label, count in zip(unique_y, y_counts):
        print(f'label = {label:<5} | count = {count:<5} | prop = {count/total_counts*100:.2f} %')

def one_hot_encode(y):
    y_ohe = np.zeros((y.size, y.max()))
    y_ohe[np.arange(y.size), y-1] = 1
    return y_ohe

In [5]:
# Distribution of labels
show_distribution(y)

label = 1     | count = 1625  | prop = 46.62 %
label = 2     | count = 233   | prop = 6.68 %
label = 3     | count = 30    | prop = 0.86 %
label = 4     | count = 483   | prop = 13.86 %
label = 5     | count = 287   | prop = 8.23 %
label = 6     | count = 310   | prop = 8.89 %
label = 7     | count = 52    | prop = 1.49 %
label = 8     | count = 466   | prop = 13.37 %


As observed, the dataset is imbalanced with label = 1 comprising almost half of the data while label = 3 is almost negligible. To solve this issue, "Synthetic Minority Oversampling Technique" (SMOTE) is applied.

# SMOTE

In [6]:
oversample = SMOTE(sampling_strategy='auto', random_state=42)
X, y = oversample.fit_resample(X, y)

In [7]:
print(f'New shapes')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

New shapes
X shape: (13000, 354)
y shape: (13000,)


In [8]:
show_distribution(y)

label = 1     | count = 1625  | prop = 12.50 %
label = 2     | count = 1625  | prop = 12.50 %
label = 3     | count = 1625  | prop = 12.50 %
label = 4     | count = 1625  | prop = 12.50 %
label = 5     | count = 1625  | prop = 12.50 %
label = 6     | count = 1625  | prop = 12.50 %
label = 7     | count = 1625  | prop = 12.50 %
label = 8     | count = 1625  | prop = 12.50 %


In [9]:
# One hot encode the labels 
y_ohe = one_hot_encode(y)
y_ohe

array([[0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.]])

# Split Dataset by Train and Validation

In [10]:
def train_test_split(X, y, train_size=0.75, random_state=42):
    """
    A simple implementation of splitting the dataset (roughly patterned after
    scikit-learn's train_test_split). It gets the indices for each class,
    divides the indices to train_indices and test_indices, according to the 
    preferred train_size, and then shuffles them.
    """
    n_samples = X.shape[0]
    indices = np.arange(n_samples)
    rng = np.random.default_rng(seed=random_state)
    rng.shuffle(indices)

    if isinstance(train_size, float):
        train_indices = indices[:int(n_samples*train_size)]
        test_indices = indices[int(n_samples*train_size):]

    elif isinstance(train_size, int):
        train_indices = indices[:train_size]
        test_indices = indices[train_size:]

    X_train = X[train_indices].copy()
    X_test = X[test_indices].copy()
    y_train = y[train_indices].copy()
    y_test = y[test_indices].copy()

    return X_train, X_test, y_train, y_test

In [14]:
X_train, X_validate, y_train, y_validate = train_test_split(
    X, y, 
    train_size=12200, 
    random_state=42
)

In [15]:
y_ohe_train = one_hot_encode(y_train)
y_ohe_validate = one_hot_encode(y_validate)

In [16]:
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_ohe_train shape: {y_ohe_train.shape}')

print(f'X_validate shape: {X_validate.shape}')
print(f'y_validate shape: {y_validate.shape}')
print(f'y_ohe_validate shape: {y_ohe_validate.shape}')

X_train shape: (12200, 354)
y_train shape: (12200,)
y_ohe_train shape: (12200, 8)
X_validate shape: (800, 354)
y_validate shape: (800,)
y_ohe_validate shape: (800, 8)


In [17]:
show_distribution(y_train)

label = 1     | count = 1504  | prop = 12.33 %
label = 2     | count = 1528  | prop = 12.52 %
label = 3     | count = 1539  | prop = 12.61 %
label = 4     | count = 1513  | prop = 12.40 %
label = 5     | count = 1516  | prop = 12.43 %
label = 6     | count = 1536  | prop = 12.59 %
label = 7     | count = 1530  | prop = 12.54 %
label = 8     | count = 1534  | prop = 12.57 %


In [18]:
show_distribution(y_validate)

label = 1     | count = 121   | prop = 15.12 %
label = 2     | count = 97    | prop = 12.12 %
label = 3     | count = 86    | prop = 10.75 %
label = 4     | count = 112   | prop = 14.00 %
label = 5     | count = 109   | prop = 13.63 %
label = 6     | count = 89    | prop = 11.12 %
label = 7     | count = 95    | prop = 11.88 %
label = 8     | count = 91    | prop = 11.38 %


# Generate Balanced Dataset

In [19]:
np.savetxt("training_set.csv", X_train, delimiter=",")
np.savetxt("training_labels.csv", y_train, delimiter=",")

In [23]:
np.savetxt("validation_set.csv", X_validate, delimiter=",")
np.savetxt("validation_labels.csv", y_validate, delimiter=",")